# 03. 발신 통화 분석 (STT + 유형 분류 + 유형별 상세 분석)

NAS에 저장된 통화 녹음 파일 중 **발신(상담원→외부) 통화**만 대상으로, STT(faster-whisper) + Ollama(qwen2.5:3b) 로컬 분류로
상담원이 건 전화를 **고객콜백 / 공급업체 / 기타** 3가지로 분류하고, 유형별로 다른 항목을 추출합니다.
수신(외부→상담원) 통화 분석은 `01_analyze_inbound_calls.ipynb`에서 다룹니다.

- **고객콜백**: 상담원이 고객에게 다시 건 콜백 → callback_reason/resolution_status/customer_sentiment/QA 추출
- **공급업체**: 공급사·제조사에 재고/가격/납기 확인 전화 → supplier_name/products/price_negotiation/order_confirmed/issues 추출
- **기타**: 잘못 건 전화, 내부 통화, 업무 무관 → 분류 사유만 기록

- **입력**: NAS 폴더(`.env`의 `VOICE_NAS_PATH`)의 `.wav`/`.mp3` 파일. 파일명 형식: `{발신}_{수신}_{시작시각}_{종료시각}.wav`. `data/call_metadata.csv`는 수신 노트북과 공용입니다.
- **출력**: `../data/outbound/` 아래 분류결과/체크포인트
- **필요 환경**: `.env`의 `VOICE_NAS_PATH`, 로컬 Ollama 서버(`qwen2.5:3b` pull 필요), `faster-whisper`
- **테스트 모드**: `TEST_N`에 숫자를 넣으면 그 개수만 처리합니다 (`None`이면 전체 처리)

In [ ]:
import os
import re
import json
from datetime import datetime

import pandas as pd
import requests
from dotenv import load_dotenv
from faster_whisper import WhisperModel

load_dotenv()

# ============================================================
# 0. 설정
# ============================================================
VOICE_NAS_PATH = os.environ.get("VOICE_NAS_PATH", "").strip()
if not VOICE_NAS_PATH:
    raise RuntimeError(
        ".env에 VOICE_NAS_PATH가 설정되지 않았습니다. "
        r"NAS 상의 통화 녹음 폴더 경로(예: \\NAS서버\mslab\DCS)를 .env에 넣어주세요."
    )

DATA_DIR = "../data/outbound"
os.makedirs(DATA_DIR, exist_ok=True)

METADATA_CSV = "../data/call_metadata.csv"
CHECKPOINT_FILE = f"{DATA_DIR}/outbound_call_checkpoint.json"
CLASSIFICATION_CSV = f"{DATA_DIR}/outbound_call_classification.csv"
CLASSIFICATION_JSON = f"{DATA_DIR}/outbound_call_classification.json"

TARGET_NUMBER = "401"
MODEL_SIZE = "small"              # tiny / base / small
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"
CPU_THREADS = 4
MAX_CHARS = 1800                 # 프롬프트 최대 글자 수
MIN_DURATION = 10                # 초 미만은 STT 스킵
SAVE_EVERY = 20                  # 몇 건마다 체크포인트 저장할지

VALID_TYPES = {"고객콜백", "공급업체", "기타"}

# 테스트 모드
# TEST_DATE: "20260601"처럼 특정 날짜(YYYYMMDD, 폴더명과 동일)만 테스트하고 싶으면 설정. None이면 날짜 제한 없음
# TEST_N: 정수를 넣으면 그 개수만 처리, None이면 전체 처리 (야간 배치 권장)
TEST_DATE = None
TEST_N = 5

In [ ]:
# ============================================================
# 1. 파일명 기반 메타데이터 파싱 (수신 노트북과 동일한 파일명 규칙, call_metadata.csv 공용)
# ============================================================
def parse_call_metadata(base_path: str) -> pd.DataFrame:
    """NAS 폴더를 순회하며 파일명에서 발신/수신, 통화시간을 추출합니다.
    파일명 형식: {발신번호}_{수신번호}_{시작YYYY_MM_DD_HH_MM_SS}_{종료YYYY_MM_DD_HH_MM_SS}.wav
    """
    records = []
    for root, _dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root)
        if not (len(folder_name) == 8 and folder_name.isdigit()):
            continue
        for fname in files:
            name, ext = os.path.splitext(fname)
            if ext.lower() not in (".wav", ".mp3"):
                continue

            parts = name.split("_")
            caller = parts[0] if len(parts) > 0 else ""
            receiver = parts[1] if len(parts) > 1 else ""

            start_dt = end_dt = duration_sec = None
            if len(parts) >= 14:
                try:
                    start_dt = datetime.strptime("_".join(parts[2:8]), "%Y_%m_%d_%H_%M_%S")
                    end_dt = datetime.strptime("_".join(parts[8:14]), "%Y_%m_%d_%H_%M_%S")
                    duration_sec = (end_dt - start_dt).total_seconds()
                except Exception:
                    pass

            records.append({
                "folder": folder_name,
                "filename": fname,
                "filepath": os.path.join(root, fname),
                "caller": caller,
                "receiver": receiver,
                "start_dt": start_dt,
                "end_dt": end_dt,
                "duration_sec": duration_sec,
                "direction": "발신" if caller == TARGET_NUMBER else "수신",
            })
    return pd.DataFrame(records)


df = parse_call_metadata(VOICE_NAS_PATH)
df.to_csv(METADATA_CSV, index=False, encoding="utf-8-sig")

print(f"총 파일: {len(df)}개")
print(df["direction"].value_counts().to_string())
print(f"\n메타데이터 저장 완료 → {METADATA_CSV}")

In [ ]:
# ============================================================
# 2. STT + Ollama 유형분류/상세분석 함수
# ============================================================
print(f"Whisper 모델 로딩 중... ({MODEL_SIZE})")
whisper_model = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8", cpu_threads=CPU_THREADS)
print("Whisper 모델 로딩 완료")

try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=5)
    available_models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Ollama 연결 OK | 사용 가능 모델: {available_models}")
    if OLLAMA_MODEL not in available_models:
        print(f"경고: {OLLAMA_MODEL} 없음 → 'ollama pull {OLLAMA_MODEL}' 실행 필요")
except Exception as e:
    print(f"Ollama 연결 실패: {e}")


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip())


def transcribe(filepath: str) -> str:
    segments, _ = whisper_model.transcribe(
        filepath, language="ko", vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )
    return normalize_text(" ".join(s.text for s in segments))


# qwen2.5는 중국어 모델 특성상 한국어 프롬프트에도 중국어가 섞여 나올 수 있어, 언어 강제 문구 + 감지 후 재시도로 방어합니다.
LANG_GUARD = (
    "!!! IMPORTANT: You MUST respond ONLY in Korean (한국어). "
    "중국어(Chinese) 사용 절대 금지. 영어도 사용하지 마세요. "
    "모든 텍스트 필드 값은 반드시 한국어로만 작성하세요. !!!"
)
HAS_CHINESE_RE = re.compile(r"[一-鿿㐀-䶿]")


def has_chinese(text: str) -> bool:
    return bool(HAS_CHINESE_RE.search(text))


# ───────────────────────────────────────────
# 1단계: 유형 분류 (고객콜백 / 공급업체 / 기타)
# ───────────────────────────────────────────
def build_type_prompt(transcript: str) -> str:
    return f"""{LANG_GUARD}

기프트코(판촉물 판매 사이트) 상담원이 외부로 건 발신 통화입니다.
반드시 한국어로만 답변하세요. Do not use Chinese or any other language.

【핵심 판단 기준: 상대방이 누구인가?】

상대방이 공급사·제조사·도매업체 직원이면 → "공급업체"
  · 기프트코에 상품을 납품하는 업체
  · 상담원이 재고·단가·납기·발주·인쇄조건 등을 확인하려 전화
  · 상대방이 "저희 상품", "저희 단가", "납기" 같은 공급자 표현 사용
  · 사이트 등록 단가·상품 정보 확인 (예: "단가 맞나요?", "등록 정보 확인")

상대방이 구매 고객·의뢰인이면 → "고객콜백"
  · 기프트코에서 상품을 사려는 일반 소비자 또는 구매 담당자
  · 상담원이 이전 문의에 답변하거나 주문·배송·결제를 처리하려 전화
  · 상대방이 "주문했는데", "언제 오나요" 같은 구매자 표현 사용

위 두 가지에 해당하지 않으면 → "기타"
  · 잘못 건 전화, 업무 무관, STT 불량, 판단 불가

규칙: JSON만 반환. 마크다운/코드블록 없이. 모든 텍스트 값은 반드시 한국어로 작성.

JSON:
{{"call_type":"고객콜백|공급업체|기타","reason":"상대방 역할 근거 한 줄"}}

전사문:
\"\"\"{transcript}\"\"\"
""".strip()


# ───────────────────────────────────────────
# 2단계 A: 고객콜백 상세 분석
# ───────────────────────────────────────────
def build_customer_prompt(transcript: str) -> str:
    return f"""{LANG_GUARD}

당신은 기프트코(판촉물 판매 사이트) 상담원의 고객 콜백 통화를 분석하는 전문가입니다.
반드시 한국어로만 답변하세요. Do not use Chinese or any other language.
기프트코는 판촉물(볼펜, 가방, 머그컵, 의류, 인쇄물 등)을 판매하는 B2B/B2C 사이트입니다.

이 통화는 상담원(기프트코)이 고객에게 직접 먼저 전화를 건 콜백입니다.

작업 목록 (모두 추출):
1. callback_reason — 상담원이 전화한 이유 (예: "배송 지연 안내", "견적 답변", "주문 확인", "결제 확인" 등)
2. resolution_status — 이번 통화에서 문제/문의가 해결됐는지 (해결/미해결/부분해결/불명확)
3. customer_sentiment — 고객의 반응/태도 (만족/중립/불만/불명확)
4. follow_up_needed — 추가 후속 조치 필요 여부 (true/false/null)
5. follow_up_detail — 후속 조치 내용 (없으면 null)
6. speaker_dialogue — 화자 구분 대화 [{{"speaker":"고객|상담원|불명","text":"내용"}}]
7. qa_pairs — 챗봇 학습용 QA 쌍. "이전 문의 → 이번 답변" 형태로 재구성. 실질적 내용만.
8. summary — 통화 전체 한 줄 요약

규칙: JSON만 반환. 마크다운/코드블록 없이. 모든 텍스트 값은 반드시 한국어로 작성.

JSON:
{{"callback_reason":"","resolution_status":"해결|미해결|부분해결|불명확","customer_sentiment":"만족|중립|불만|불명확","follow_up_needed":null,"follow_up_detail":null,"speaker_dialogue":[{{"speaker":"고객|상담원|불명","text":"내용"}}],"qa_pairs":[{{"question":"질문","answer":"답변"}}],"summary":"요약"}}

전사문:
\"\"\"{transcript}\"\"\"
""".strip()


# ───────────────────────────────────────────
# 2단계 B: 공급업체 통화 상세 분석
# ───────────────────────────────────────────
def build_supplier_prompt(transcript: str) -> str:
    return f"""{LANG_GUARD}

당신은 기프트코(판촉물 판매 사이트) 상담원과 공급업체 간의 통화를 분석하는 전문가입니다.
반드시 한국어로만 답변하세요. Do not use Chinese or any other language.
기프트코는 판촉물(볼펜, 가방, 머그컵, 의류, 인쇄물 등)을 판매하며, 상담원이 공급업체에 직접 전화해 재고·가격·납기 등을 확인합니다.

아래 항목을 모두 추출하세요.

1. supplier_name: 공급업체·업체명 (언급 없으면 null)
2. summary: 통화 전체 한 줄 요약
3. products: 언급된 상품 목록 (배열). 각 항목:
   - product_name: 상품명
   - quantity: 수량 (언급 없으면 null)
   - unit_price: 단가 (언급 없으면 null)
   - total_price: 총액 (언급 없으면 null)
   - stock_available: 재고 여부 (true/false/null)
   - delivery_date: 납기·배송 예정 (언급 없으면 null)
   - print_condition: 인쇄·제작 조건 (언급 없으면 null)
   - notes: 기타 특이사항
4. price_negotiation: 가격 협의 내용 요약 (없으면 null)
5. order_confirmed: 발주 확정 여부 (true/false/null)
6. issues: 문제점·이슈 (재고 없음, 납기 지연, 가격 이견 등. 없으면 null)
7. follow_up: 후속 조치 필요 사항 (없으면 null)
8. speaker_dialogue: 화자 구분 대화 [{{"speaker":"상담원|공급업체담당|불명","text":"내용"}}]

규칙: JSON만 반환. 마크다운/코드블록 없이. 모든 텍스트 값은 반드시 한국어로 작성.

JSON:
{{"supplier_name":null,"summary":"","products":[{{"product_name":"","quantity":null,"unit_price":null,"total_price":null,"stock_available":null,"delivery_date":null,"print_condition":null,"notes":""}}],"price_negotiation":null,"order_confirmed":null,"issues":null,"follow_up":null,"speaker_dialogue":[{{"speaker":"상담원|공급업체담당|불명","text":"내용"}}]}}

전사문:
\"\"\"{transcript}\"\"\"
""".strip()


def safe_parse_json(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e > s:
        try:
            return json.loads(text[s:e + 1])
        except Exception:
            pass
    return {}


def normalize_call_type(raw: str) -> str:
    raw = (raw or "").strip()
    if raw in VALID_TYPES:
        return raw
    if "고객" in raw:
        return "고객콜백"
    if "공급" in raw:
        return "공급업체"
    return "기타"


def call_ollama(prompt: str, max_retries: int = 2) -> dict:
    """중국어 응답이 감지되면 최대 max_retries회 재시도합니다 (qwen2.5 계열의 알려진 이슈)."""
    payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "format": "json"}
    result, raw = {}, ""
    for attempt in range(max_retries + 1):
        resp = requests.post(OLLAMA_URL, json=payload, timeout=(30, 600))
        resp.raise_for_status()
        raw = resp.json().get("response", "")
        result = safe_parse_json(raw)
        if not has_chinese(raw):
            return result
        if attempt < max_retries:
            print(f"         ⚠ 중국어 감지 → 재시도 ({attempt + 1}/{max_retries})")
    print("         ⚠ 재시도 후에도 중국어 포함 — 결과 그대로 사용")
    return result


def classify_call_type(transcript: str) -> tuple[str, str]:
    parsed = call_ollama(build_type_prompt(transcript[:MAX_CHARS]))
    return normalize_call_type(parsed.get("call_type", "")), parsed.get("reason", "")


def analyze_customer_callback(transcript: str) -> dict:
    parsed = call_ollama(build_customer_prompt(transcript[:MAX_CHARS]))
    dialogue = [d for d in parsed.get("speaker_dialogue", [])
                if isinstance(d, dict) and d.get("text")]
    qa_pairs = [q for q in parsed.get("qa_pairs", [])
                if isinstance(q, dict) and q.get("question") and q.get("answer")]
    return {
        "summary": parsed.get("summary", ""),
        "callback_reason": parsed.get("callback_reason", ""),
        "resolution_status": parsed.get("resolution_status", "불명확"),
        "customer_sentiment": parsed.get("customer_sentiment", "불명확"),
        "follow_up_needed": parsed.get("follow_up_needed"),
        "follow_up_detail": parsed.get("follow_up_detail"),
        "speaker_dialogue": dialogue,
        "qa_pairs": qa_pairs,
        # 공급업체 전용 필드는 빈값
        "supplier_name": None,
        "products": [],
        "price_negotiation": None,
        "order_confirmed": None,
        "issues": None,
        "follow_up": None,
    }


def analyze_supplier_call(transcript: str) -> dict:
    parsed = call_ollama(build_supplier_prompt(transcript[:MAX_CHARS]))
    dialogue = [d for d in parsed.get("speaker_dialogue", [])
                if isinstance(d, dict) and d.get("text")]
    products = [p for p in parsed.get("products", [])
                if isinstance(p, dict) and p.get("product_name")]
    return {
        "summary": parsed.get("summary", ""),
        "speaker_dialogue": dialogue,
        "supplier_name": parsed.get("supplier_name"),
        "products": products,
        "price_negotiation": parsed.get("price_negotiation"),
        "order_confirmed": parsed.get("order_confirmed"),
        "issues": parsed.get("issues"),
        "follow_up": parsed.get("follow_up"),
        # 고객콜백 전용 필드는 빈값
        "callback_reason": "",
        "resolution_status": "",
        "customer_sentiment": "",
        "follow_up_needed": None,
        "follow_up_detail": None,
        "qa_pairs": [],
    }


print("함수 정의 완료")

In [ ]:
# ============================================================
# 3. 발신 통화 분류 배치 실행 (체크포인트 이어받기 + 테스트 모드 지원)
# ============================================================
def classify_outbound_calls():
    outbound_df = df[(df["direction"] == "발신") & (df["caller"] == TARGET_NUMBER)].copy()

    if TEST_DATE:
        outbound_df = outbound_df[outbound_df["folder"] == TEST_DATE]
        print(f"[날짜 필터] {TEST_DATE} 폴더만 대상 → {len(outbound_df)}개")

    target_df = outbound_df[outbound_df["duration_sec"] >= MIN_DURATION].copy()
    skipped = len(outbound_df) - len(target_df)

    print(f"전체 파일: {len(df)}개 (수신 {len(df[df['direction']=='수신'])}개 / 발신 {len(df[df['direction']=='발신'])}개)")
    print(f"필터 후 대상: {len(outbound_df)}개 | 처리 대상: {len(target_df)}개 | {MIN_DURATION}초 미만 스킵: {skipped}개")

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            ckpt = json.load(f)
        done_files = set(ckpt.get("done_files", []))
        all_results = ckpt.get("results", [])
        print(f"체크포인트 복원: {len(done_files)}개 완료")
    else:
        done_files = set()
        all_results = []
        print("새로 시작")

    todo_df = target_df[~target_df["filepath"].isin(done_files)]

    if TEST_N:
        todo_df = todo_df.head(TEST_N)
        print(f"[테스트 모드] {TEST_N}건만 처리 (전체 배치는 TEST_N=None으로 변경 후 실행)")

    start_time = datetime.now()
    total = len(todo_df)

    for i, (_, row) in enumerate(todo_df.iterrows(), 1):
        filepath = row["filepath"]

        # NAS 연결이 끊겨 파일을 못 찾는 경우: 완료 처리하지 않고 건너뛰어 다음 실행 때 재시도되게 함
        if not os.path.exists(filepath):
            print(f"[파일없음 스킵] {row['filename']} — 네트워크 연결 확인 후 재실행하세요.")
            continue

        result_row = {
            "file_name": row["filename"], "filepath": filepath, "folder": row["folder"],
            "duration_sec": row["duration_sec"],
            "call_type": "", "call_type_reason": "", "summary": "", "transcript": "",
            "speaker_dialogue_text": "", "qa_pairs_text": "", "qa_count": 0,
            "speaker_dialogue": [], "qa_pairs": [],
            "callback_reason": "", "resolution_status": "", "customer_sentiment": "",
            "follow_up_needed": None, "follow_up_detail": None,
            "supplier_name": None, "products": [], "product_count": 0,
            "price_negotiation": None, "order_confirmed": None, "issues": None, "follow_up": None,
        }

        try:
            transcript = transcribe(filepath)
        except Exception as e:
            transcript = ""
            print(f"[STT 에러] {row['filename']}: {e}")
        result_row["transcript"] = transcript

        if not transcript:
            result_row["call_type"] = "기타"
            result_row["call_type_reason"] = "STT 전사 없음"
            all_results.append(result_row)
            done_files.add(filepath)
        else:
            try:
                call_type, ct_reason = classify_call_type(transcript)
            except Exception as e:
                call_type, ct_reason = "기타", f"분류 에러: {e}"
                print(f"[유형분류 에러] {row['filename']}: {e}")

            result_row["call_type"] = call_type
            result_row["call_type_reason"] = ct_reason

            try:
                if call_type == "고객콜백":
                    analysis = analyze_customer_callback(transcript)
                elif call_type == "공급업체":
                    analysis = analyze_supplier_call(transcript)
                else:
                    analysis = {}
                if analysis:
                    products = analysis.get("products", [])
                    result_row.update(analysis)
                    result_row["product_count"] = len(products)
                    result_row["speaker_dialogue_text"] = "\n".join(
                        f"{d['speaker']}: {d['text']}" for d in analysis["speaker_dialogue"]
                    )
                    result_row["qa_pairs_text"] = "\n\n".join(
                        f"[QA {j}]\nQ: {q['question']}\nA: {q['answer']}"
                        for j, q in enumerate(analysis["qa_pairs"], 1)
                    )
                    result_row["qa_count"] = len(analysis["qa_pairs"])
            except Exception as e:
                print(f"[분석 에러] {row['filename']}: {e}")

            all_results.append(result_row)
            done_files.add(filepath)

        elapsed = (datetime.now() - start_time).total_seconds()
        if i % 5 == 0 or i == total:
            remaining = elapsed / i * (total - i)
            print(f"[{i:4d}/{total}] {result_row['call_type']:<10} | 경과 {elapsed/60:.1f}분 | "
                  f"남은예상 {remaining/60:.1f}분 | {row['filename'][:40]}")

        if i % SAVE_EVERY == 0 or i == total:
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump({"done_files": list(done_files), "results": all_results}, f, ensure_ascii=False)
            pd.DataFrame(all_results).to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")

    result_df = pd.DataFrame(all_results)
    result_df.to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")
    result_df.to_json(CLASSIFICATION_JSON, orient="records", force_ascii=False, indent=2)
    print(f"\n완료 → {CLASSIFICATION_CSV}")
    return result_df


result_df = classify_outbound_calls()

In [ ]:
# ============================================================
# 4. 결과 요약
# ============================================================
result_df = pd.read_csv(CLASSIFICATION_CSV, encoding="utf-8-sig")

print("=" * 50)
print("[ 발신 통화 유형별 분류 결과 ]")
print("=" * 50)
type_counts = result_df["call_type"].value_counts()
for call_type, cnt in type_counts.items():
    print(f"  {call_type:<10}: {cnt:>5}개 ({cnt / len(result_df) * 100:.1f}%)")
print(f"  {'합계':<10}: {len(result_df):>5}개")

callbacks = result_df[result_df["call_type"] == "고객콜백"]
suppliers = result_df[result_df["call_type"] == "공급업체"]

print(f"\n고객콜백: {len(callbacks)}개 | 생성된 QA: {callbacks['qa_count'].sum() if len(callbacks) else 0}개")
print(f"공급업체: {len(suppliers)}개 | 상품 언급: {suppliers['product_count'].sum() if len(suppliers) else 0}건 | "
      f"발주확정: {(suppliers['order_confirmed'] == True).sum() if len(suppliers) else 0}건")

print(f"\n결과 파일: {CLASSIFICATION_CSV}")
print(f"           {CLASSIFICATION_JSON}")